In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from glob import glob

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "figure.figsize": (5, 4),
})


In [ ]:
# ==========================================
# 1. Helper functions
# ==========================================

base_dir = "./"

def discover_temperatures(base_dir="./"):
    summary = os.path.join(base_dir, "summary_all.csv")
    if os.path.exists(summary):
        df = pd.read_csv(summary)
        return np.sort(df['T'].dropna().unique())
    vals = []
    for path in glob(os.path.join(base_dir, "T_*")):
        try:
            vals.append(float(os.path.basename(path).split("_", 1)[1]))
        except ValueError:
            pass
    return np.array(sorted(vals))

def find_T_dir(T, base_dir="./"):
    candidates = [f"T_{T}", f"T_{T:g}", f"T_{T:.3f}", f"T_{T:.2f}"]
    for name in candidates:
        path = os.path.join(base_dir, name)
        if os.path.isdir(path):
            return path
    matches = glob(os.path.join(base_dir, f"T_{T:.3f}".rstrip('0').rstrip('.') + "*"))
    return matches[0] if matches else os.path.join(base_dir, f"T_{T:g}")

def read_dos(data_dir, filename="spectra_dos.csv"):
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return None
    df = pd.read_csv(file_path)
    if df.empty:
        print(f"Warning: Empty file {file_path}")
        return None
    return df.sort_values('omega')

def get_dos_at_omega0(df, value_col='DOS', err_col='DOS_Error'):
    omega = df['omega'].to_numpy()
    vals = df[value_col].to_numpy()
    idx0 = int(np.argmin(np.abs(omega)))
    err0 = np.nan
    if err_col in df.columns:
        err0 = df[err_col].to_numpy()[idx0]
    return vals[idx0], err0, float(omega[idx0])


In [ ]:
# ==========================================
# 2. Main loop
# ==========================================

T_list = discover_temperatures(base_dir)
T_list = T_list[(T_list >= 0.005) & (T_list <= 0.100)]

results = {"T": [], "N0": [], "N0_err": [], "M0": [], "M0_err": []}
spectra = []

print(f"Starting DOS analysis for {len(T_list)} temperatures...")

for T in T_list:
    data_dir = find_T_dir(T, base_dir)
    print(f"Processing {os.path.basename(data_dir)}...", end='\r')
    df = read_dos(data_dir)
    if df is None:
        continue

    omega = df['omega'].to_numpy()
    dos = df['DOS'].to_numpy()
    dos_err = df['DOS_Error'].to_numpy() if 'DOS_Error' in df.columns else None
    spectra.append({'T': T, 'omega': omega, 'dos': dos, 'dos_err': dos_err})

    n0, n0_err, omega0 = get_dos_at_omega0(df, 'DOS', 'DOS_Error')
    results['T'].append(T)
    results['N0'].append(n0)
    results['N0_err'].append(n0_err)

    if 'DOS_M' in df.columns:
        m0, m0_err, _ = get_dos_at_omega0(df, 'DOS_M', 'DOS_M_Error')
    else:
        m_file = read_dos(data_dir, 'spectra_dos_M.csv')
        if m_file is not None:
            m0, m0_err, _ = get_dos_at_omega0(m_file, 'DOS_M', 'Error')
        else:
            m0, m0_err = np.nan, np.nan
    results['M0'].append(m0)
    results['M0_err'].append(m0_err)

print()
print("Analysis complete.")
for key in results:
    results[key] = np.array(results[key])


In [ ]:
# ==========================================
# 3. N(omega) vs omega for different T
# ==========================================

fig, ax = plt.subplots(dpi=300)
colors = plt.cm.viridis(np.linspace(0, 1, len(spectra)))

for i, spec in enumerate(spectra):
    ax.plot(spec['omega'], spec['dos'], color=colors[i], label=rf"T={spec['T']:g}")

ax.set_xlabel(r'$\omega$')
ax.set_ylabel(r'$N(\omega)$')
ax.legend(frameon=False, ncol=2)
plt.show()


In [ ]:
# ==========================================
# 4. N(omega=0) and M-point spectrum vs T
# ==========================================

fig, ax = plt.subplots(dpi=300)
mask = np.isfinite(results['T']) & np.isfinite(results['N0'])
ax.errorbar(results['T'][mask], results['N0'][mask], yerr=results['N0_err'][mask],
            fmt='-o', color='black', capsize=3, label=r'$N(0)$')
mask_m = np.isfinite(results['T']) & np.isfinite(results['M0'])
if np.any(mask_m):
    ax.errorbar(results['T'][mask_m], results['M0'][mask_m], yerr=results['M0_err'][mask_m],
                fmt='-s', color='tab:orange', capsize=3, label=r'$A_M(0)$')
ax.set_xlabel(r'$T$')
ax.set_ylabel(r'spectral weight at $\omega=0$')
ax.set_xlim(0.0, 0.105)
ax.legend(frameon=False)
plt.show()


In [ ]:
# ==========================================
# 5. dN(omega=0)/dT vs T
# ==========================================

t_vals = results['T'].copy()
n0_vals = results['N0'].copy()

mask = np.isfinite(t_vals) & np.isfinite(n0_vals)
t_vals = t_vals[mask]
n0_vals = n0_vals[mask]

sort_idx = np.argsort(t_vals)
t_vals = t_vals[sort_idx]
n0_vals = n0_vals[sort_idx]

if len(t_vals) < 2:
    print("Not enough points to compute dN/dT.")
else:
    dN_dT = np.gradient(n0_vals, t_vals)

    fig, ax = plt.subplots(dpi=300)
    ax.plot(t_vals, dN_dT, '-o', color='blue')
    ax.set_xlabel(r'$T$')
    ax.set_ylabel(r'$dN(\omega=0)/dT$')
    plt.show()
